In [83]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# Ignoro i warning 
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Percorso da cui prendere il file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Serve per far paritre il debug
DEBUG = False

# Carico il csv e stampo la shape

In [84]:
df_duke = pd.read_csv(FILE_PATH / "duke_lesions.csv")

print("DUKE shape: ", df_duke.shape)

DUKE shape:  (291, 109)


# Target

In [85]:
marker = ["PR", "ER"]

# Bilancio il dataset in fase di training

In [86]:
def undersample_training_set(X, y, random_state=42):
    df_tmp = pd.concat([X, y], axis=1)

    df_0 = df_tmp[df_tmp[y.name] == 0]
    df_1 = df_tmp[df_tmp[y.name] == 1]

    n_min = min(len(df_0), len(df_1))

    df_bal = pd.concat([
        df_0.sample(n=n_min, random_state=random_state),
        df_1.sample(n=n_min, random_state=random_state)
    ])

    X_bal = df_bal.drop(columns=[y.name])
    y_bal = df_bal[y.name]

    return X_bal, y_bal


# Controllo il numero delle classi

In [87]:
for m in marker:
    vc = df_duke[m].value_counts(dropna=False)

    print(f"\nDistribuzione {m} – DUKE")
    print("-" * 30)
    print("Negativi:", vc.get(0, 0))
    print("Positivi:", vc.get(1, 0))
    print("Totale pazienti:", len(df_duke) )



Distribuzione PR – DUKE
------------------------------
Negativi: 157
Positivi: 134
Totale pazienti: 291

Distribuzione ER – DUKE
------------------------------
Negativi: 123
Positivi: 168
Totale pazienti: 291


# Stratified Cross-Validation

In [88]:
FEATURES = [c for c in df_duke.columns if c.startswith("original_")]

# Training

In [89]:
results_rows = []
for target in marker:

    print("\n" + "="*80)
    print(f"TARGET: {target}")
    print("="*80)

    X = df_duke[FEATURES]
    y = df_duke[target]

    print("Distribuzione originale:", dict(y.value_counts()))

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    acc_scores, bal_scores, f1_scores, auc_scores = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

        # split reale
        X_train_raw, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train_raw, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # undersampling SOLO train
        X_train, y_train = undersample_training_set(
            X_train_raw, y_train_raw
        )

        print(f"\n--- Fold {fold} ---")
        print("Supporto TRAIN (bilanciato):", dict(y_train.value_counts()))
        print("Supporto TEST (reale):", dict(y_test.value_counts()))
        

        model = XGBClassifier(
            random_state=42,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        bal = balanced_accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)

        acc_scores.append(acc)
        bal_scores.append(bal)
        f1_scores.append(f1)
        auc_scores.append(auc)

        print(f"Accuracy: {acc:.4f} | Balanced Accuracy: {bal:.4f}")
        print(classification_report(y_test, y_pred, zero_division=0))

    # =======================
    # RISULTATI FINALI
    # =======================

    print("\n" + "-"*50)
    print(f"RISULTATI MEDI FINALI - {target}")
    print(f"Accuracy : {np.mean(acc_scores):.3f} ± {np.std(acc_scores):.3f}")
    print(f"Balanced : {np.mean(bal_scores):.3f} ± {np.std(bal_scores):.3f}")
    print(f"F1-score : {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
    print(f"ROC-AUC  : {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}")
    print("-"*50)



    results_rows.append({
        "Dataset": "DUKE",
        "Target": target,
        "Accuracy": f"{np.mean(acc_scores):.3f} ± {np.std(acc_scores):.3f}",
        "Balanced Accuracy": f"{np.mean(bal_scores):.3f} ± {np.std(bal_scores):.3f}",
        "F1-score": f"{np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}",
        "ROC-AUC": f"{np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}"
    })
    
results_df = pd.DataFrame(results_rows)
results_df.to_csv("XGBoost_balanced_training_results.csv", index=False)
print("\nRisultati salvati in XGBoost_balanced_training_results.csv")




TARGET: PR
Distribuzione originale: {0: np.int64(157), 1: np.int64(134)}

--- Fold 1 ---
Supporto TRAIN (bilanciato): {0: np.int64(107), 1: np.int64(107)}
Supporto TEST (reale): {0: np.int64(32), 1: np.int64(27)}
Accuracy: 0.5085 | Balanced Accuracy: 0.5093
              precision    recall  f1-score   support

           0       0.55      0.50      0.52        32
           1       0.47      0.52      0.49        27

    accuracy                           0.51        59
   macro avg       0.51      0.51      0.51        59
weighted avg       0.51      0.51      0.51        59


--- Fold 2 ---
Supporto TRAIN (bilanciato): {0: np.int64(108), 1: np.int64(108)}
Supporto TEST (reale): {0: np.int64(32), 1: np.int64(26)}
Accuracy: 0.3793 | Balanced Accuracy: 0.3834
              precision    recall  f1-score   support

           0       0.42      0.34      0.38        32
           1       0.34      0.42      0.38        26

    accuracy                           0.38        58
   macro av

# Controllo il numero delle classi

In [90]:
for m in marker:
    vc_bal = y.value_counts()
    print("Negativi (classe 0):", vc_bal.get(0, 0))
    print("Positivi (classe 1):", vc_bal.get(1, 0))
    print(f"Totale pazienti usati: {len(y)}")


Negativi (classe 0): 123
Positivi (classe 1): 168
Totale pazienti usati: 291
Negativi (classe 0): 123
Positivi (classe 1): 168
Totale pazienti usati: 291
